# Late Fusion System for Multi-Feature Model Integration
---
This notebook implements a **Late Fusion** framework for combining predictions from multiple machine learning models trained on different feature sets:

- **Gas & Environmental features**
- **Specter features**
- **Combined (All)** features
- **Image-based** features

The system automatically selects the **best-performing model** for each feature type, allows the user to select one or more feature groups, and performs **late fusion** to combine the final predictions.

---

## Import Required Libraries

In [1]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model

## Define the Best Models per Feature Type

In [2]:
best_models = {
    'gas_env': {
        'name': 'Gradient Boost',
        'path': 'gas&env_models/gradient_boost_model_s.pkl',
        'type': 'sklearn'
    },
    'specter': {
        'name': 'Random Forest',
        'path': 'specter_models/random_forest_model_s.pkl',
        'type': 'sklearn'
    },
    'all': {
        'name': 'Gradient Boost',
        'path': 'all_models/gradient_boost_model_s.pkl',
        'type': 'sklearn'
    },
    'image': {
        'name': 'CNN (Image Model)',
        'path': 'image_classification_model.h5',
        'type': 'keras'
    }
}

## Define Helper Functions for Model Loading and Prediction

In [3]:
def load_best_model(feature_type):
    """Loads the best model for a given feature type."""
    model_info = best_models[feature_type]
    path = model_info['path']
    
    if model_info['type'] == 'sklearn':
        with open(path, 'rb') as f:
            model = pickle.load(f)
    elif model_info['type'] == 'keras':
        model = load_model(path)
    else:
        raise ValueError(f"Unsupported model type for {feature_type}")
    
    return model, model_info['name']


def get_model_prediction(model, x_input):
    """Returns model predictions (probabilities or logits)."""
    if hasattr(model, 'predict_proba'):  # sklearn-like models
        pred = model.predict_proba(x_input)
    else:
        pred = model.predict(x_input)
        if len(pred.shape) == 1:
            pred = np.expand_dims(pred, axis=1)
    return pred

## Implement the Late Fusion Mechanism

In [4]:
def late_fusion(selected_features, feature_inputs):
    """Combines predictions from the best models of each selected feature type."""
    predictions = []
    used_models = []

    for ftype in selected_features:
        model, model_name = load_best_model(ftype)
        pred = get_model_prediction(model, feature_inputs[ftype])
        predictions.append(pred)
        used_models.append((ftype, model_name))

    predictions = np.array(predictions)

    if len(predictions) == 1:
        fused = predictions[0]
    else:
        fused = np.mean(predictions, axis=0)

    return fused, used_models